# 04 – Model Evaluation & Explainability
**Project:** War Economic Impact Predictor  

## Objectives
1. Comprehensive test-set metric reporting  
2. Residual analysis for regression model  
3. Confusion matrix & per-class metrics for classifier  
4. SHAP global and local explainability  
5. Error analysis — which conflicts are hardest to predict?

---

In [ ]:
import sys, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import joblib
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score,
    mean_absolute_percentage_error,
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    roc_auc_score, f1_score,
)
from sklearn.model_selection import train_test_split

warnings.filterwarnings('ignore')
shap.initjs()

ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT))
from src.utils import load_config

%matplotlib inline
plt.rcParams['figure.dpi'] = 120
cfg = load_config(ROOT / 'config' / 'config.yaml')
RS = cfg['data']['random_state']
print('Setup OK')

In [ ]:
df = pd.read_parquet(ROOT / 'data' / 'processed' / 'war_economic_features.parquet')
TARGET_REG = cfg['data']['target_regression']
TARGET_CLS = cfg['data']['target_classification']
EXCLUDE = {TARGET_REG, TARGET_CLS}
feat_cols = [c for c in df.columns if c not in EXCLUDE]

X = df[feat_cols].fillna(0)
y_reg = df[TARGET_REG]
y_cls = df[TARGET_CLS]

X_tr, X_te, yr_tr, yr_te = train_test_split(X, y_reg, test_size=0.2, random_state=RS)
_, _, yc_tr, yc_te = train_test_split(X, y_cls, test_size=0.2, random_state=RS, stratify=y_cls)

model_dir = ROOT / 'models'
reg_model  = joblib.load(model_dir / 'regression_xgb.joblib')
cls_model  = joblib.load(model_dir / 'classification_xgb.joblib')
scaler     = joblib.load(model_dir / 'scaler_xgb.joblib')

X_te_sc = scaler.transform(X_te)
yr_pred = reg_model.predict(X_te_sc)
yc_pred = cls_model.predict(X_te_sc)

print(f'Test set size: {len(X_te):,}')

## 1. Regression Metrics

In [ ]:
rmse = np.sqrt(mean_squared_error(yr_te, yr_pred))
mae  = mean_absolute_error(yr_te, yr_pred)
mape = mean_absolute_percentage_error(yr_te, yr_pred)
r2   = r2_score(yr_te, yr_pred)

metrics = pd.Series({'RMSE': rmse, 'MAE': mae, 'MAPE': mape, 'R²': r2})
print(metrics.round(4).to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Actual vs Predicted
ax = axes[0]
ax.scatter(yr_te, yr_pred, alpha=0.25, s=12, color='steelblue')
lo, hi = min(yr_te.min(), yr_pred.min()), max(yr_te.max(), yr_pred.max())
ax.plot([lo, hi], [lo, hi], 'r--', lw=1.5, label='Perfect fit')
ax.set_xlabel('Actual GDP Change (%)')
ax.set_ylabel('Predicted GDP Change (%)')
ax.set_title(f'Actual vs Predicted  (R²={r2:.3f})')
ax.legend()

# Residuals
resid = yr_te.values - yr_pred
ax2 = axes[1]
ax2.scatter(yr_pred, resid, alpha=0.25, s=12, color='coral')
ax2.axhline(0, color='black', lw=1)
ax2.set_xlabel('Predicted')
ax2.set_ylabel('Residual')
ax2.set_title(f'Residual Plot  (RMSE={rmse:.3f})')

plt.tight_layout()
plt.savefig(ROOT / 'reports' / 'figures' / 'regression_evaluation.png', dpi=150)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(resid, bins=50, kde=True, color='teal', ax=ax)
ax.axvline(0, color='red', ls='--')
ax.set_xlabel('Residual')
ax.set_title('Residual Distribution')
plt.tight_layout()
plt.show()

## 2. Classification Metrics

In [ ]:
labels = ['Mild', 'Moderate', 'Severe', 'Catastrophic']
print(classification_report(yc_te, yc_pred, target_names=labels))

In [ ]:
cm = confusion_matrix(yc_te, yc_pred)
fig, ax = plt.subplots(figsize=(7, 6))
disp = ConfusionMatrixDisplay(cm, display_labels=labels)
disp.plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title('Confusion Matrix – Economic Severity Classifier')
plt.tight_layout()
plt.savefig(ROOT / 'reports' / 'figures' / 'confusion_matrix_xgb.png', dpi=150)
plt.show()

## 3. SHAP Explainability

In [ ]:
X_te_df = pd.DataFrame(X_te_sc, columns=feat_cols)
sample = X_te_df.sample(min(1000, len(X_te_df)), random_state=RS)

explainer = shap.TreeExplainer(reg_model)
shap_vals = explainer.shap_values(sample)

# Global summary (bar)
plt.figure(figsize=(10, 7))
shap.summary_plot(shap_vals, sample, plot_type='bar', max_display=20, show=False)
plt.title('SHAP – Global Feature Importance (Regression)')
plt.tight_layout()
plt.savefig(ROOT / 'reports' / 'figures' / 'shap_global_bar.png', dpi=150)
plt.show()

In [ ]:
# Beeswarm (magnitude + direction)
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_vals, sample, max_display=20, show=False)
plt.title('SHAP Beeswarm – Feature Impact on GDP Change (%)')
plt.tight_layout()
plt.savefig(ROOT / 'reports' / 'figures' / 'shap_beeswarm.png', dpi=150)
plt.show()

In [ ]:
# Local explanation for a single prediction (worst predicted instance)
worst_idx = np.argmax(np.abs(resid[:len(sample)]))
shap.force_plot(
    explainer.expected_value,
    shap_vals[worst_idx],
    sample.iloc[worst_idx],
    matplotlib=True,
    show=False,
    figsize=(16, 3),
)
plt.title('Local SHAP – Worst Prediction')
plt.tight_layout()
plt.savefig(ROOT / 'reports' / 'figures' / 'shap_local.png', dpi=150)
plt.show()

## 4. Error Analysis

In [ ]:
error_df = X_te.copy()
error_df['Actual_GDP'] = yr_te.values
error_df['Predicted_GDP'] = yr_pred
error_df['Abs_Residual'] = np.abs(resid)

# Worst predictions
print('Top 10 largest errors:')
display(error_df.nlargest(10, 'Abs_Residual')[['Actual_GDP', 'Predicted_GDP', 'Abs_Residual']].reset_index(drop=True))

In [ ]:
# Error by true severity class
error_df['True_Severity'] = yc_te.values
label_map = {0: 'Mild', 1: 'Moderate', 2: 'Severe', 3: 'Catastrophic'}
error_df['Severity_Name'] = error_df['True_Severity'].map(label_map)

mean_err_by_sev = error_df.groupby('Severity_Name')['Abs_Residual'].mean().reindex(label_map.values())

fig, ax = plt.subplots(figsize=(7, 4))
mean_err_by_sev.plot(kind='bar', color=['#2196F3','#FF9800','#F44336','#B71C1C'], ax=ax)
ax.set_ylabel('Mean Absolute Residual')
ax.set_title('Mean Prediction Error by Severity Class')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
plt.tight_layout()
plt.savefig(ROOT / 'reports' / 'figures' / 'error_by_severity.png', dpi=150)
plt.show()

## 5. Evaluation Summary

### Regression (GDP Change % prediction)
| Metric | Value |
|--------|-------|
| RMSE | *run to populate* |
| MAE | *run to populate* |
| R² | *run to populate* |

### Classification (Severity Label)
| Metric | Value |
|--------|-------|
| F1-Weighted | *run to populate* |
| F1-Macro | *run to populate* |

### Key SHAP findings
- Top predictors: *inflation, currency devaluation, unemployment spike, poverty rate*  
- Interaction features (e.g., `Unemployment_Spike_multiply_Conflict_Duration`) added explanatory power  
- Catastrophic cases are systematically harder — longer war duration compounds model error

> **Next step:** Deploy via `app/app.py` (Streamlit)